In [11]:
print("test")

test


In [6]:
import sys
from pathlib import Path

# Raíz del repo: notebooks/ -> subir un nivel
ROOT = Path.cwd().resolve()
if (ROOT / "src" / "dspy_lab").is_dir():
    SRC = ROOT / "src"
elif (ROOT.parent / "src" / "dspy_lab").is_dir():
    SRC = ROOT.parent / "src"  # si el cwd es notebooks/
else:
    raise RuntimeError("No encuentro src/dspy_lab")

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [10]:
import os
import importlib
import dspy_lab.lm_config as dspy_lm_config

os.environ["DSPY_LM_BACKEND"] = "ollama"
os.environ["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"  # notebook local Windows
# os.environ["OLLAMA_MODEL"] = "llama3.2:latest"
os.environ["OLLAMA_MODEL"] = "phi3:latest"

importlib.reload(dspy_lm_config)

from dspy_lab.lm_config import configure_lm
import dspy

lm = configure_lm(max_tokens=1024)
print(lm.model)

print(dspy.Predict("tema: str -> idea: str")(tema="RAG con Ollama").idea)

ollama_chat/phi3:latest
"Envision a digital RAG (Rendered Advertising Generation) platform fused seamlessly with Ollama's cutting-edge language model, providing creators unprecedented access to text generation and personalization. This hybrid tool enables the crafting of unique, captivating advertisements that resonate deeply with target audiences through dynamic narrative creation."
[ [ ## completed ## ] ]


In [12]:
configure_lm(max_tokens=1024)

qa = dspy.Predict("pregunta: str -> respuesta: str")
r = qa(pregunta="En una frase, Que es DSPy?")
print(r.respuesta)



DSPy significa un "Desarrollador de Software para Diseño Musical". Es una herramienta desarrollada por el equipo Korg en Japón, que permite a los usuarios crear y diseñar sus propios temas musicales utilizando instrumentos virtuales. La naturaleza interactiva del software facilita la experimentación con diferentes texturas sonoras y ritmos para producir composiciones originales o simular el trabajo de otros artistas. DSPy se destaca por su interfaz accesible e intuitiva, que a menudo es un atractivo tanto para músicos novatos como para profesionales establecidos en la industria del entretenimiento musical. El software incluye una variedad rica de instrumentos virtuales y permite efectos avanzados, tales como moduladores y filtros sonoros digitales no nativos, lo que ofrece un amplio espectro creativo para el diseño auditivo en la era digital.


## Firmas explícitas (más legibles)


In [14]:
import dspy

class Resumen(dspy.Signature):
    """Resume el texto en español, máximo 2 oraciones."""
    texto: str = dspy.InputField()
    resumen: str = dspy.OutputField()

configure_lm(max_tokens=512)
resumir = dspy.Predict(Resumen)
out = resumir(texto="DSPy es un framework para programar con LLMs en lugar de solo escribir prompts.")
print(out.resumen)

# ]]
    DSPy facilita programar con modelos grandes mediante una interfaz basada en texto, mejorando su accesibilidad y usabilidad para desarrolladores sin experiencia previa en técninqas avanzadas de IA.


## ChainOfThought (razonar antes de responder)

In [16]:
class QA(dspy.Signature):
    pregunta: str = dspy.InputField()
    respuesta: str = dspy.OutputField(desc="Una frase correcta en español.")

cot = dspy.ChainOfThought(QA)
r = cot(pregunta="¿Qué es DSPy (Stanford NLP framework)?")
print("razonamiento:", r.reasoning)
print("respuesta:", r.respuesta)

razonamiento: Para crear una respuesta precisa y en español, primero necesitban investigar sobre el término "DSPy", que podría referirse a un acrónimo o nombre propietario específico dentro del contexto de la lingüística computacional. Dado que no existe información ampliamente conocida sobre este términino hasta mi fecha límite, suena más probable que se trate de una versión localizada u optimizada para el español y las necesidades particulares del proyecto o institución en cuestión (en este caso, asociadas con la Universidad de Stanford).
respuesta: DSPy es un marco específico adaptado por la comunidad española para aplicaciones de procesamiento de lenguaje natural que se desea personalizar a través del uso y contribución en proyectos académicos o empresariales, con una ampliación al ámbito localizado.


## Módulo reutilizable (dspy.Module)

In [17]:
class AsistenteQA(dspy.Module):
    def __init__(self):
        self.generar = dspy.ChainOfThought("pregunta: str -> respuesta: str")

    def forward(self, pregunta: str):
        return self.generar(pregunta=pregunta)

configure_lm(max_tokens=1024)
asistente = AsistenteQA()
print(asistente(pregunta="¿Para qué sirve dspy.Predict?").respuesta)

Dspy.Predict es una función potencial dentro de la biblioteca DSPy que permite realizar predicciones basadas en computaciones simbólicas dinámicas y numéricas, especialmente útil para aplicaciones educativas enfocadas en el aprendizaje reinforcioso. Con ella, los estudiantes pueden introducir variables a un modelo matemático o físico y obtener retroalimentación inmediata sobre resultados predichos que reflejen escenarios hipotéticos o concretos basados en las leyes naturales representadas por el software.


## Pipeline en dos pasos (idea de agente simple)

In [18]:
class Idea(dspy.Signature):
    tema: str = dspy.InputField()
    idea: str = dspy.OutputField()

class Titulo(dspy.Signature):
    idea: str = dspy.InputField()
    titulo: str = dspy.OutputField(desc="Título corto para un post.")

class PipelineBlog(dspy.Module):
    def __init__(self):
        self.ideas = dspy.Predict(Idea)
        self.titulos = dspy.Predict(Titulo)

    def forward(self, tema: str):
        idea = self.ideas(tema=tema).idea
        titulo = self.titulos(idea=idea).titulo
        return dspy.Prediction(idea=idea, titulo=titulo)

p = PipelineBlog()(tema="RAG con Ollama")
print(p.idea)
print(p.titulo)

"Envision a digital RAG (Rendered Advertising Generation) platform fused seamlessly with Ollama's cutting-edge language model, providing creators unprecedented access to text generation and personalization. This hybrid tool enables the crafting of unique, captivating advertisements that resonate deeply with target audiences through dynamic narrative creation."
[ [ ## completed ## ] ]
Generación de Texto Digital para Creaciones de Publicidad Innovadoras


## Evaluación con métrica simple (núcleo de DSPy)

In [19]:
trainset = [
    dspy.Example(pregunta="¿Qué es DSPy?", respuesta_esperada="framework").with_inputs("pregunta"),
    dspy.Example(pregunta="¿Qué es un LM?", respuesta_esperada="modelo de lenguaje").with_inputs("pregunta"),
]

def metrica(example, pred, trace=None):
    # Heurística barata: la respuesta menciona palabras clave
    texto = (pred.respuesta or "").lower()
    clave = example.respuesta_esperada.lower()
    return clave in texto

qa = dspy.Predict("pregunta: str -> respuesta: str")
evaluador = dspy.Evaluate(devset=trainset, metric=metrica, num_threads=1, display_progress=True)
evaluador(qa)

Average Metric: 1.00 / 2 (50.0%): 100%|██████████| 2/2 [00:05<00:00,  2.95s/it]

2026/05/22 01:25:32 INFO dspy.evaluate.evaluate: Average Metric: 1 / 2 (50.0%)


EvaluationResult(score=50.0, results=<list of 2 results>)

## Optimización con pocos ejemplos (BootstrapFewShot)

In [20]:
from dspy.teleprompt import BootstrapFewShot

teleprompter = BootstrapFewShot(metric=metrica, max_bootstrapped_demos=3, max_labeled_demos=3)
qa_optimizado = teleprompter.compile(qa, trainset=trainset)

evaluador(qa_optimizado)

100%|██████████| 2/2 [00:02<00:00,  1.03s/it]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Average Metric: 1.00 / 2 (50.0%): 100%|██████████| 2/2 [00:05<00:00,  2.68s/it]

2026/05/22 01:26:14 INFO dspy.evaluate.evaluate: Average Metric: 1 / 2 (50.0%)


EvaluationResult(score=50.0, results=<list of 2 results>)

##  Inspeccionar qué se envió al modelo

In [21]:
dspy.inspect_history(n=1)  # última llamada al LM





[2026-05-22T01:26:14.530992]

System message:

Your input fields are:
1. `pregunta` (str):
Your output fields are:
1. `respuesta` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## pregunta ## ]]
{pregunta}

[[ ## respuesta ## ]]
{respuesta}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `pregunta`, produce the fields `respuesta`.


User message:

[[ ## pregunta ## ]]
¿Qué es un LM?


Assistant message:

[[ ## respuesta ## ]]
Un LM, o modelo de lenguaje, es una tecnología que permite a las computadoras generar texto similar al humano.

[[ ## completed ## ]]


User message:

[[ ## pregunta ## ]]
¿Qué es un LM?

Respond with the corresponding output fields, starting with the field `[[ ## respuesta ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## respuesta ## ]]
Un LM, o modelo de lenguaje, es una tecnología que permite a las computador